# From Messy Data to a Deployed Model — One End-to-End Notebook

Today we build ONE project, start to finish, in this order:

1. **Data Preprocessing Pipelines** — clean messy data automatically and safely
2. **Model Evaluation & Cross-Validation** — check if our model is actually good, honestly
3. **Hyperparameter Tuning** — squeeze out better settings with Grid Search and Random Search
4. **Deploying with Streamlit** — turn the finished model into a real web app

Every step builds on the one before it, so run the cells top to bottom with **Shift + Enter**. Every line of code has a comment explaining exactly what it does.

## Step 1 — Import Everything We'll Need Today

In [12]:
# numpy and pandas: our usual tools for numbers and tables
import numpy as np
import pandas as pd

# train_test_split: splits data into a training portion and a test portion
from sklearn.model_selection import train_test_split

# these three build our automatic cleaning pipeline
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# the model we'll train, tune, and deploy today
from sklearn.ensemble import RandomForestClassifier

# tools for Part 2: honestly evaluating the model
from sklearn.model_selection import StratifiedKFold, cross_val_score

# tools for Part 3: automatically searching for better settings
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

# joblib: lets us save a trained pipeline to a file, to load again later (e.g. inside our app)
import joblib

# time: just so we can compare how long different searches take
import time

# lock in the "randomness" so everyone gets the exact same numbers every time
np.random.seed(42)

# a simple message to confirm every import above worked with no errors
print("Ready to go!")

Ready to go!


## Step 2 — Build a Realistically MESSY Dataset

Real data is never clean. We'll simulate 300 students with 4 features, and then deliberately punch some holes in it (missing values) and add a text category — exactly the kind of mess a real pipeline has to handle automatically.

In [13]:
# how many students to simulate
n_students = 300

# hours_studied: most students cluster around 5 hours, some study much more or less
hours_studied = np.clip(np.random.normal(5, 2.3, n_students), 0, 10).round(1)

# attendance_pct: how much of class they attended
attendance_pct = np.clip(np.random.normal(75, 18, n_students), 0, 100).round(0)

# previous_score: their score on the last test
previous_score = np.clip(np.random.normal(65, 15, n_students), 0, 100).round(0)

# study_method: a CATEGORY column (text, not a number) -- this is why we need encoding later
study_method = np.random.choice(["Group", "Solo", "Online"], size=n_students, p=[0.4, 0.4, 0.2])

# combine the numeric features into a rough "pass score", weighted so hours_studied matters most
combined_score = 0.5 * (hours_studied / 10) + 0.3 * (attendance_pct / 100) + 0.2 * (previous_score / 100)

# add randomness so the pattern isn't perfectly clean
noisy_score = combined_score + np.random.normal(0, 0.13, n_students)

# a student "passes" (1) if their noisy score clears 0.5, otherwise they "fail" (0)
passed = (noisy_score > 0.5).astype(int)

# assemble everything into one table
students = pd.DataFrame({
    "hours_studied": hours_studied,
    "attendance_pct": attendance_pct,
    "previous_score": previous_score,
    "study_method": study_method,
    "passed": passed
})

# now deliberately punch holes in the data, like a real messy dataset would have
# each of these picks a random slice of rows and blanks out that one column
for column, missing_fraction in [("hours_studied", 0.08), ("attendance_pct", 0.05), ("previous_score", 0.08), ("study_method", 0.10)]:
    rows_to_blank = students.sample(frac=missing_fraction, random_state=1).index
    students.loc[rows_to_blank, column] = np.nan

# show how many missing values ended up in each column
print("Missing values per column:")
print(students.isna().sum())

# display the first 10 rows so we can see the messiness for ourselves (look for NaN values)
students.head(10)

Missing values per column:
hours_studied     24
attendance_pct    15
previous_score    24
study_method      30
passed             0
dtype: int64


,hours_studied,attendance_pct,previous_score,study_method,passed
0,6.1,60.0,76.0,Group,1
1,4.7,65.0,51.0,Online,1
2,6.5,88.0,78.0,Online,1
3,8.5,86.0,85.0,Solo,1
4,4.5,75.0,71.0,Group,1
5,4.5,77.0,93.0,Group,1
6,8.6,98.0,53.0,Solo,1
7,6.8,64.0,46.0,Online,1
8,3.9,85.0,38.0,Solo,1
9,6.2,71.0,87.0,Group,1


# Part 1 — Data Preprocessing Pipelines

We can't feed missing values or text categories straight into a model. Normally you'd write separate clean-up code for each column and hope you remember to apply it identically to both training and test data. A **Pipeline** does this automatically, in one object, so it's impossible to forget a step or apply it inconsistently.

In [14]:
# split the data BEFORE we do any cleaning -- this is important.
# if we cleaned first and split after, information from the test set could leak into training (data leakage)
X = students.drop(columns=["passed"])   # every column except the answer
y = students["passed"]                  # the answer column

# test_size=0.25 -- hold out 25% of rows for testing, keep 75% for training
# random_state=42 -- makes this exact split reproducible every time we run the cell
# stratify=y -- keeps the pass/fail ratio the same in both the train and test portions
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# print how many rows ended up in each portion, just to sanity-check the split
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

Training rows: 225
Test rows: 75


In [15]:
# list which columns are numbers -- these will go through the numeric_pipeline below
numeric_features = ["hours_studied", "attendance_pct", "previous_score"]

# list which column(s) are text categories -- these will go through the categorical_pipeline below
categorical_features = ["study_method"]

# NUMERIC pipeline: first fill in missing values with the column's median, then scale everything
# to a similar range (important for many models, and harmless for the ones that don't need it)
numeric_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler())
])

# CATEGORICAL pipeline: fill missing values with the most common category, then one-hot encode
# (turn "Group"/"Solo"/"Online" into separate 0/1 columns the model can actually read)
categorical_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("encode", OneHotEncoder(handle_unknown="ignore"))
])

# ColumnTransformer applies the RIGHT pipeline to the RIGHT columns, all in one step
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

# confirm the preprocessor object was built successfully (nothing has been trained yet)
print("Preprocessor built!")

Preprocessor built!


In [16]:
# now chain the preprocessor and the model together into ONE single pipeline object.
# calling .fit() on this one object will clean AND train in one go, in the correct order
model_pipeline = Pipeline([
    ("prep", preprocessor),
    ("clf", RandomForestClassifier(random_state=42))
])

# train the whole pipeline -- cleaning happens automatically before the model ever sees the data
model_pipeline.fit(X_train, y_train)

# score it on the untouched test set (this cleans the test set with the SAME rules learned from training, then checks accuracy)
baseline_accuracy = model_pipeline.score(X_test, y_test)

# print the accuracy as a percentage, rounded to 1 decimal place
print(f"Baseline test accuracy: {baseline_accuracy:.1%}")

Baseline test accuracy: 81.3%


**Why this matters:** that one `model_pipeline` object now contains the entire workflow — imputing, scaling, encoding, and predicting. You can save it, reload it, and hand it a single messy row of new data, and it'll clean and predict in one call. That's exactly what we'll do in Part 4.

# Part 2 — Model Evaluation & Cross-Validation

The accuracy above came from just ONE train/test split. But what if we got a lucky (or unlucky) split? A single number can't tell us that. **Cross-validation** answers this by splitting the data multiple different ways and checking how much the score actually moves around.

In [17]:
# StratifiedKFold splits the data into 5 folds, keeping the pass/fail ratio balanced in every fold
cv_splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# cross_val_score automatically: trains on 4 folds, tests on the 5th, and repeats 5 times,
# rotating which fold is held out each time -- returning one score per fold
fold_scores = cross_val_score(model_pipeline, X_train, y_train, cv=cv_splitter)

# print all 5 individual scores, rounded to 3 decimal places, so we can see them bounce around
print("Score on each of the 5 folds:", fold_scores.round(3))

# the average of the 5 folds -- a much more trustworthy accuracy estimate than any single split
print(f"Mean accuracy: {fold_scores.mean():.1%}")

# how much the folds vary from each other -- a small number means a stable, reliable model
print(f"Standard deviation: {fold_scores.std():.1%}")

Score on each of the 5 folds: [0.667 0.8   0.711 0.822 0.756]
Mean accuracy: 75.1%
Standard deviation: 5.7%


**Reading this:** notice the 5 fold scores aren't identical — they genuinely bounce around. That spread is the whole point of cross-validation: it tells you how much to actually trust your accuracy number, instead of reporting a single score that might just be luck.

# Part 3 — Hyperparameter Tuning (Grid Search vs. Random Search)

Our RandomForest has settings we haven't touched yet — `n_estimators` (how many trees), `max_depth` (how deep each tree can grow), and more. These are called **hyperparameters**, and finding good values for them can meaningfully improve the model.

**Grid Search** tries EVERY combination you give it. **Random Search** tries a random sample of combinations instead — usually almost as good, in a fraction of the time.

In [ ]:
# GRID SEARCH: define an exact, small grid of values to try every combination of
param_grid = {
    "clf__n_estimators": [100, 200],
    "clf__max_depth": [4, 8, None],
    "clf__min_samples_split": [2, 5]
}
# note the "clf__" prefix -- it tells the pipeline "this setting belongs to the step named 'clf'"

# record the current time, so we can measure how long the whole search takes below
start_time = time.time()

# GridSearchCV wraps our pipeline and will train + cross-validate it once for EVERY combination
# in param_grid -- cv=cv_splitter reuses the same 5-fold splitting strategy from Part 2,
# and n_jobs=-1 tells it to use all available CPU cores to run combinations in parallel
grid_search = GridSearchCV(model_pipeline, param_grid, cv=cv_splitter, n_jobs=-1)

# .fit() actually runs the entire search: every combination, every fold, then keeps the best one
grid_search.fit(X_train, y_train)

# subtract the start time from now to get how many seconds the search took
grid_search_seconds = time.time() - start_time

# multiply the number of choices for each setting together, to show the total combinations tried
print(f"Grid Search checked {len(param_grid['clf__n_estimators']) * len(param_grid['clf__max_depth']) * len(param_grid['clf__min_samples_split'])} combinations")

# best_score_ is the highest cross-validated accuracy found across every combination tried
print(f"Best cross-validated accuracy: {grid_search.best_score_:.1%}")

# best_params_ shows exactly which settings produced that best score
print("Best settings found:", grid_search.best_params_)

# print how long the search took, rounded to 2 decimal places
print(f"Time taken: {grid_search_seconds:.2f} seconds")

In [ ]:
# RANDOM SEARCH: define a WIDER range of values, but only sample a fixed number of combinations
param_distribution = {
    "clf__n_estimators": [50, 100, 150, 200, 300],
    "clf__max_depth": [3, 4, 5, 6, 8, 10, None],
    "clf__min_samples_split": [2, 3, 5, 8, 10],
    "clf__min_samples_leaf": [1, 2, 4]
}
# this grid has 5 x 7 x 5 x 3 = 525 possible combinations -- far too many to try all of

# record the current time again, so we can time this search too
start_time = time.time()

# RandomizedSearchCV works like GridSearchCV, but instead of trying every combination,
# it randomly samples a fixed number of combinations from the space we gave it
random_search = RandomizedSearchCV(
    model_pipeline, param_distribution,
    n_iter=10,              # only try 10 random combinations, not all 525
    cv=cv_splitter,         # reuse the same 5-fold cross-validation strategy as before
    random_state=42,        # lock in WHICH 10 random combinations get picked, for reproducibility
    n_jobs=-1                # use all available CPU cores to run combinations in parallel
)

# .fit() runs the search: 10 combinations x 5 folds each, then keeps the best combination found
random_search.fit(X_train, y_train)

# measure how long this search took, the same way we did for grid search
random_search_seconds = time.time() - start_time

# remind ourselves how small a slice of the total space we actually searched
print("Random Search checked only 10 combinations out of 525 possible")

# the highest cross-validated accuracy found among the 10 combinations tried
print(f"Best cross-validated accuracy: {random_search.best_score_:.1%}")

# exactly which settings produced that best score
print("Best settings found:", random_search.best_params_)

# print how long the search took, rounded to 2 decimal places
print(f"Time taken: {random_search_seconds:.2f} seconds")

**Compare the two:** Random Search searched a much bigger space of options, in less time, and landed on a similarly good result. This is exactly why Random Search is usually preferred once your hyperparameter grid gets large — Grid Search becomes too slow to be practical.

## Step — Lock In Our Final Model

In [ ]:
# grid_search and random_search both automatically kept their best pipeline, fully trained,
# ready to use immediately -- no need to retrain by hand
final_model = grid_search.best_estimator_

# evaluate it one last time on the test set we set aside all the way back in Part 1
final_test_accuracy = final_model.score(X_test, y_test)

# print both numbers side by side so they're easy to compare
print(f"Baseline (untuned) test accuracy: {baseline_accuracy:.1%}")
print(f"Final (tuned) test accuracy:      {final_test_accuracy:.1%}")

**If the tuned score isn't higher than the baseline, that's OK — and expected.** Our test set only has ~75 rows, so a single flipped prediction shifts the accuracy by more than a percentage point. Tuning optimized for the best AVERAGE score across 5 cross-validation folds (a much more reliable signal), not for this one specific test split. This is exactly why we did cross-validation in Part 2 in the first place — trust the multi-fold average over any single number.

# Part 4 — Deploying with Streamlit

Streamlit turns a plain Python script into a working web app — no HTML, CSS, or JavaScript needed. The plan: save our finished pipeline to a file, then write a small app that loads it and makes live predictions from user input.

In [ ]:
# joblib.dump saves our ENTIRE pipeline -- imputers, scaler, encoder, and trained model -- into one file.
# anything that loads this file later can clean and predict on brand new data immediately
joblib.dump(final_model, "student_pass_predictor.joblib")

# confirm the save worked and remind ourselves of the exact filename we'll load later
print("Model saved to student_pass_predictor.joblib")

The `%%writefile` line below is a special Colab/Jupyter command — it doesn't run as normal Python. Instead, it takes everything else in this cell and saves it as a real file called `app.py`, sitting right alongside your notebook. This file is the actual Streamlit app; it does not run inside the notebook itself.

In [ ]:
%%writefile app.py
# app.py -- a small Streamlit web app that loads our saved pipeline and makes live predictions.
# Run it from a terminal (not from this notebook) with:  streamlit run app.py

# streamlit: the library that turns this script into a web app
import streamlit as st

# pandas: to build a one-row table from the user's input, the same shape our pipeline expects
import pandas as pd

# joblib: to load the pipeline we saved from the notebook
import joblib

# load our saved pipeline once when the app starts
model = joblib.load("student_pass_predictor.joblib")

# a title shown at the top of the web page
st.title("Will This Student Pass?")

# a plain line of instructional text, shown just below the title
st.write("Enter a student's details below, and the model will predict pass or fail.")

# st.slider(label, minimum, maximum, default) -- creates a draggable slider on the page
# and returns whatever number the user has it set to, every time the page updates
hours_studied = st.slider("Hours studied", 0.0, 10.0, 5.0)
attendance_pct = st.slider("Attendance percent", 0.0, 100.0, 75.0)
previous_score = st.slider("Previous test score", 0.0, 100.0, 65.0)

# st.selectbox(label, options) -- creates a dropdown menu, and returns whichever option is chosen
study_method = st.selectbox("Study method", ["Group", "Solo", "Online"])

# a button -- the code below only runs when the user clicks it
if st.button("Predict"):

    # build a one-row DataFrame with the exact same column names the pipeline was trained on
    input_row = pd.DataFrame([{
        "hours_studied": hours_studied,
        "attendance_pct": attendance_pct,
        "previous_score": previous_score,
        "study_method": study_method
    }])

    # the pipeline cleans (impute/scale/encode) AND predicts, in one call
    prediction = model.predict(input_row)[0]

    # predict_proba gives us the model's confidence, not just its final answer
    probability = model.predict_proba(input_row)[0][1]

    # show a big, clear result on the page
    if prediction == 1:
        # st.success shows a green box -- used here for a PASS prediction
        st.success(f"Prediction: PASS (confidence: {probability:.0%})")
    else:
        # st.error shows a red box -- used here for a FAIL prediction (not a code error!)
        st.error(f"Prediction: FAIL (confidence: {1 - probability:.0%})")

**To actually run this app:** download `app.py` and `student_pass_predictor.joblib` from the Colab file panel, put them in the same folder on your computer, install Streamlit once with `pip install streamlit`, then run:

```
streamlit run app.py
```

A browser tab will open automatically with your working prediction app.

## Recap

- **Pipelines** bundle every cleaning step (imputing, scaling, encoding) and the model into one object — safer and impossible to apply inconsistently.
- **Cross-validation** trains and tests on several different splits, so you see how much your accuracy score actually moves around, instead of trusting one lucky split.
- **Grid Search** tries every combination you specify. **Random Search** samples a fixed number of combinations from a wider space — usually just as good, much faster.
- A trained pipeline can be saved with `joblib` and loaded straight into a **Streamlit** app, turning it into a real, usable tool in well under 50 lines of code.

That's a full project lifecycle: messy data in, working web app out.

In [ ]:
# Instead of median, try mean imputation for numeric columns
numeric_pipeline_mean = Pipeline([
    ("impute", SimpleImputer(strategy="mean")),   # changed from median
    ("scale", StandardScaler())
])

# Rebuild the pipeline and re‑run the training/evaluation...

In [ ]:
from sklearn.linear_model import LogisticRegression

# Replace RandomForest with LogisticRegression in the pipeline
model_pipeline_lr = Pipeline([
    ("prep", preprocessor),
    ("clf", LogisticRegression(random_state=42, max_iter=1000))
])

model_pipeline_lr.fit(X_train, y_train)
print("Logistic Regression test accuracy:", model_pipeline_lr.score(X_test, y_test))

In [ ]:
for n in [3, 5, 10]:
    cv = StratifiedKFold(n_splits=n, shuffle=True, random_state=42)
    scores = cross_val_score(model_pipeline, X_train, y_train, cv=cv)
    print(f"{n} folds: mean = {scores.mean():.1%}, std = {scores.std():.1%}")

In [ ]:
estimators = [10, 50, 100, 200, 500]
scores = []
for n in estimators:
    pipe = Pipeline([
        ("prep", preprocessor),
        ("clf", RandomForestClassifier(n_estimators=n, random_state=42))
    ])
    pipe.fit(X_train, y_train)
    scores.append(pipe.score(X_test, y_test))

plt.plot(estimators, scores, marker='o')
plt.xlabel("Number of trees (n_estimators)")
plt.ylabel("Test Accuracy")
plt.title("Effect of Forest Size")
plt.grid(True)
plt.show()

In [ ]:
# Save
joblib.dump(model_pipeline, "my_pipeline.joblib")

# Load back
loaded_pipeline = joblib.load("my_pipeline.joblib")

# Test that it still works
print("Loaded pipeline accuracy:", loaded_pipeline.score(X_test, y_test))